In [53]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("../models/ashaar_finetuned_final_8")
model = AutoModelForCausalLM.from_pretrained("../models/ashaar_finetuned_final_8")

In [54]:
import json 
theme_to_token = json.load(open("../pre_trained_models/ashaar/tokens/theme_tokens.json", "r"))
token_to_theme = {t:m for m,t in theme_to_token.items()}

meter_to_token = json.load(open("../pre_trained_models/ashaar/tokens/meter_tokens.json", "r"))
token_to_meter = {t:m for m,t in meter_to_token.items()}

In [128]:
theme = "قصيدة غزل"
meter = "الطويل"
qafiyah = "ر"
prompt = f"{meter_to_token[meter]} {qafiyah} {theme_to_token[theme]}"
print(prompt)
encoded_input = tokenizer(prompt, return_tensors='pt')
output = model.generate(**encoded_input, max_length = 156,temperature=0.5, top_p = 3, do_sample=True)
print(encoded_input)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


<|meter_13|> ر <|theme_8|>
{'input_ids': tensor([[20, 93, 53, 32]]), 'attention_mask': tensor([[1, 1, 1, 1]])}


In [129]:
result = ""
prev_token = ""

for i, beam in enumerate(output[:, len(encoded_input.input_ids[0]):]):
    for token in beam:
        # print(decoded)
        decoded = tokenizer.decode(token)
        if 'meter' in decoded or 'theme' in decoded:
            break
        if decoded == "<|endoftext|>":
            break
        if decoded == "<|vsep|>":
            result += "\n\n"
        elif decoded in ["<|bsep|>", "</|bsep|>"]:
            result += "\n"
        elif decoded in ['<|psep|>', '</|psep|>']:
            pass
        else:
            result += decoded
        prev_token = decoded
    else:
        break
print(theme+" "+ f"من بحر {meter} مع قافية بحر ({qafiyah})" + "\n" +result)

قصيدة غزل من بحر الطويل مع قافية بحر (ر)

لعمرك ني بالمدامة والكاس

وني لمشتاق لى الراح والراس

وني لمخلوق لدي لغيرها

وني لمخلوق لدي لمن ناس



In [35]:
# ...existing code...
import torch

# ensure pad/eos exist
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.eos_token is None:
    tokenizer.eos_token = "</s>"

theme = "قصيدة غزل"
meter = "الكامل"
qafiyah = "ب"
# build prompt from parts so tokenizer doesn't merge or drop tokens
enc_meter = tokenizer(meter_to_token[meter], add_special_tokens=False)["input_ids"]
enc_qaf = tokenizer(qafiyah, add_special_tokens=False)["input_ids"]
enc_theme = tokenizer(theme_to_token[theme], add_special_tokens=False)["input_ids"]
input_ids = enc_meter + enc_qaf + enc_theme

print("parts ids:", enc_meter, enc_qaf, enc_theme)
print("combined input length:", len(input_ids))

device = next(model.parameters()).device
input_tensor = torch.tensor([input_ids], device=device)

safe_pad = tokenizer.pad_token_id or tokenizer.eos_token_id or 0

# generate enough tokens for ~4 verses (tune max_new_tokens if needed)
output = model.generate(
    input_ids=input_tensor,
    max_new_tokens=512,
    do_sample=True,
    top_p=20,
    temperature=0.1,
    pad_token_id=safe_pad,
    eos_token_id=tokenizer.eos_token_id
)

# robust handling if no new tokens
if output.ndim == 1:
    output = output.unsqueeze(0)
if output.shape[1] <= len(input_ids):
    print("No new tokens generated.")
else:
    gen_ids = output[0, len(input_ids):].tolist()
    result = ""
    verse_count = 1  # count verses; we already have prompt metadata so counting generated verses
    for tid in gen_ids:
        token_str = tokenizer.convert_ids_to_tokens(tid)
        # handle special tokens as markers
        if token_str in ("<|vsep|>", "<verse>"):
            result += "\n\n"
            verse_count += 1
            if verse_count > 4:
                break
            continue
        if token_str in ("<|bsep|>", "</|bsep|>"):
            result += "\n"
            continue
        if token_str in ("<|psep|>", "</|psep|>"):
            continue
        # decode normal token id to text fragment
        print(text)
        text = tokenizer.decode([tid], skip_special_tokens=True, clean_up_tokenization_spaces=True)
        result += text
        # stop when EOS appears
        if tid == tokenizer.eos_token_id:
            break

    header = f"{theme} من بحر {meter} مع قافية ({qafiyah})"
    print(header + "\n" + result.strip())
# ...existing

parts ids: [21] [84] [32]
combined input length: 3

ي
ا
 
م
ن
 
ي
ر
ى
 
ا
ل
أ
م
ل
 
ا
ل
ب
ع
ي
د
 
ب
ب
ا
ب
ه
 
و
ا
ل
م
و
ت
 
ي
ق
ر
ب
 
م
ن
 
م
ن
ا
ه
 
ل
ى
 
م
ن
ا
ه
 
و
م
ن
 
ي
ر
ى
 
ا
ل
م
و
ت
 
ا
ل
ز
ؤ
ا
م
 
ب
ع
ز
م
ه
 
ف
ل
ي
س
 
ي
ر
د
 
م
ن
ه
 
ل
ى
 
م
ن
ا
ه
قصيدة غزل من بحر الكامل مع قافية (ب)
يا من يرى الأمل البعيد ببابه والموت يقرب من مناه لى مناه ومن يرى الموت الزؤام بعزمه فليس يرد منه لى مناه


In [357]:
def diagnose_condition_obedience(model, tokenizer):
    """Test if model actually uses your conditions"""
    
    # Test 1: Same prompt, different meters
    prompts = [
        ("<psep><meter_0><theme_0>", "meter_0"),
        ("<psep><meter_5><theme_0>", "meter_5"),
        ("<psep><meter_10><theme_0>", "meter_10"),
    ]
    
    for prompt, expected_meter in prompts:
        input_ids = torch.tensor([tokenizer.encode(prompt)])
        with torch.no_grad():
            outputs = model.generate(
                input_ids,
                max_new_tokens=50,
                temperature=0.3,
                do_sample=False,  # Greedy to see what model "prefers"
            )
        generated = tokenizer.decode(outputs[0])
        print(f"Prompt: {expected_meter}")
        print(f"Output: {generated[:100]}...")
        print("-" * 40)
    
    # Test 2: Same meter, different qafiyah
    prompts = [
        "<psep><meter_0><qaf>يا</qaf>",
        "<psep><meter_0><qaf>ون</qaf>",
    ]
    
    for prompt in prompts:
        input_ids = torch.tensor([tokenizer.encode(prompt)])
        with torch.no_grad():
            outputs = model.generate(input_ids, max_new_tokens=50, temperature=0.3)
        generated = tokenizer.decode(outputs[0])
        print(f"Qafiyah prompt: {prompt}")
        print(f"Output ends with: {generated[-30:]}")
        print("-" * 40)

In [383]:
# Quick verify tokenizer/model consistency
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained("../models/ashaar_finetuned_final_4")
m = AutoModelForCausalLM.from_pretrained("../models/ashaar_finetuned_final_4")
print("special tokens:", {t: tok.convert_tokens_to_ids(t) for t in ["<verse>","<|vsep|>","<|bsep|>","</|bsep|>","<|psep|>","</|psep|>"]})
prompt = f"{meter_to_token[meter]} {qafiyah} {theme_to_token[theme]}"
print("prompt str:", prompt)
parts = [meter_to_token[meter], qafiyah, theme_to_token[theme]]
parts_ids = [tok(p, add_special_tokens=False)["input_ids"] for p in parts]
print("parts ids:", parts_ids)
combined = sum(parts_ids, [])
print("combined ids:", combined)
# compare to one training example tokenization you printed earlier

Loading weights: 100%|██████████| 125/125 [00:00<00:00, 26925.23it/s]

Loading weights: 100%|██████████| 125/125 [00:00<00:00, 26925.23it/s]

special tokens: {'<verse>': 122, '<|vsep|>': 3, '<|bsep|>': 4, '</|bsep|>': 5, '<|psep|>': 1, '</|psep|>': 2}
prompt str: <|meter_14|> ب <|theme_8|>
parts ids: [[21], [], [32]]
combined ids: [21, 32]


In [8]:

# filepath: /home/hussam/Desktop/senior_project_github/poem_generation/generation_model_testing_2.ipynb
# build prompt robustly and debug tokenization
theme = "قصيدة غزل"
meter = "الكامل"
qafiyah = "ب"

# canonical prompt string (same format used in training)
prompt_str = f"{meter_to_token[meter]} {qafiyah} {theme_to_token[theme]}"
print("prompt string:", prompt_str)

# preferred: tokenize the whole prompt string (guarantees same tokenization as training)
enc_whole = tokenizer(prompt_str, add_special_tokens=False)["input_ids"]
print("tokenized (whole) ids:", enc_whole)

# optional: build from parts but with fallback if a part tokenizes to []
enc_meter = tokenizer(meter_to_token[meter], add_special_tokens=False)["input_ids"]
enc_qaf = tokenizer(qafiyah, add_special_tokens=False)["input_ids"]
enc_theme = tokenizer(theme_to_token[theme], add_special_tokens=False)["input_ids"]

# fallback: if single-char qaf tokenizes to empty, try with a leading space
if len(enc_qaf) == 0:
    enc_qaf = tokenizer(" " + qafiyah, add_special_tokens=False)["input_ids"]
    print("fallback enc_qaf (with leading space):", enc_qaf)

print("parts ids:", enc_meter, enc_qaf, enc_theme)
combined = enc_meter + enc_qaf + enc_theme
print("combined ids:", combined)

# use enc_whole (or combined) consistently for generation
input_ids = enc_whole  # use enc_whole to match training
import torch
device = next(model.parameters()).device
input_tensor = torch.tensor([input_ids], device=device)

safe_pad = tokenizer.pad_token_id or tokenizer.eos_token_id or 0
out = model.generate(
    input_ids=input_tensor,
    max_new_tokens=256,
    do_sample=True,
    top_p=0.95,
    temperature=0.9,
    pad_token_id=safe_pad,
    eos_token_id=tokenizer.eos_token_id,
)

if out.ndim == 1:
    out = out.unsqueeze(0)
gen_ids = out[0, len(input_ids):].tolist()
# decode while respecting your verse separators and stop after 4 verses
result = ""
verse_count = 0
for tid in gen_ids:
    token_str = tokenizer.convert_ids_to_tokens(tid)
    if token_str in ("<|vsep|>", "<verse>"):
        verse_count += 1
        if verse_count >= 4:
            break
        result += "\n\n"
        continue
    if token_str in ("<|bsep|>", "</|bsep|>"):
        result += "\n"
        continue
    if token_str in ("<|psep|>", "</|psep|>"):
        continue
    result += tokenizer.decode([tid], skip_special_tokens=True, clean_up_tokenization_spaces=True)
    if tid == tokenizer.eos_token_id:
        break

print(result.strip())


prompt string: <|meter_14|> ب <|theme_8|>
tokenized (whole) ids: [21, 84, 53, 32]
parts ids: [21] [84] [32]
combined ids: [21, 84, 32]
وَأَغَرَّ أَعطافِ السُيوفِ مُهَنَّدٍ يَفري الجُفونَ بِهِ صُدورُ المَشرَفي وَمُهَنَّدٍ يَفري العَدوَّ فَريغُهُ في مُلتَقى الِمهالِ وَالجَبّارِ
